In [1]:
# Célula 1: Importações
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from bs4 import BeautifulSoup
import re
import time
import pandas as pd
import numpy as np

In [2]:
# Célula 2: Configuração do webdriver
chrome_service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=chrome_service)

In [3]:
# Célula 3: URL
url = 'https://www.quintoandar.com.br/alugar/imovel/sao-paulo-sp-brasil/apartamento'

In [4]:
# Célula 4: Carregamento da página
driver.get(url)
print("Página de listagem carregando... Aguardando 5 segundos.")
time.sleep(5)

Página de listagem carregando... Aguardando 5 segundos.


In [5]:
# Célula 5: Fase 1 - Coleta de URLs dos imóveis

target_imoveis = 2000 # Seu alvo
segundos_espera_clique = 5 

urls_encontradas = set() # Usamos um 'set' para evitar URLs duplicadas

print(f"Iniciando Fase 1: Coletando URLs de ~{target_imoveis} imóveis...")

while len(urls_encontradas) < target_imoveis:
    try:
        # 1. Encontra e clica no botão
        ver_mais_button = driver.find_element(By.ID, "see-more")
        ver_mais_button.click()
        time.sleep(segundos_espera_clique)
        
        # 2. Pega o HTML da página atualizada
        temp_soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # 3. Encontra os links
        # Usamos o seletor de endereço que validamos ('h2', class_='_72Hu5c')
        # e subimos até a tag 'a' (link) que o envolve
        
        endereco_tags = temp_soup.find_all('h2', class_='_72Hu5c')
        
        for tag in endereco_tags:
            link_tag = tag.find_parent('a') # Procura a tag <a> "pai"
            
            if link_tag and 'href' in link_tag.attrs:
                url_parcial = link_tag['href']
                if url_parcial.startswith('/imovel'):
                    # Monta a URL completa
                    url_completa = "https://www.quintoandar.com.br" + url_parcial
                    urls_encontradas.add(url_completa)

        print(f"   ... {len(urls_encontradas)} URLs únicas coletadas ...")
            
    except NoSuchElementException:
        print("Botão 'Ver mais' não encontrado. Chegamos ao fim da página.")
        break
    except Exception as e:
        print(f"Ocorreu um erro ao tentar clicar ou extrair links: {e}")
        break

urls_para_visitar = list(urls_encontradas)
print(f"\nFase 1 Concluída. Total de {len(urls_para_visitar)} URLs prontas para Nível 2.")

Iniciando Fase 1: Coletando URLs de ~2000 imóveis...
   ... 23 URLs únicas coletadas ...
   ... 35 URLs únicas coletadas ...
   ... 47 URLs únicas coletadas ...
   ... 59 URLs únicas coletadas ...
   ... 71 URLs únicas coletadas ...
   ... 83 URLs únicas coletadas ...
   ... 95 URLs únicas coletadas ...
   ... 107 URLs únicas coletadas ...
   ... 119 URLs únicas coletadas ...
   ... 131 URLs únicas coletadas ...
   ... 143 URLs únicas coletadas ...
   ... 155 URLs únicas coletadas ...
   ... 167 URLs únicas coletadas ...
   ... 179 URLs únicas coletadas ...
   ... 191 URLs únicas coletadas ...
   ... 203 URLs únicas coletadas ...
   ... 215 URLs únicas coletadas ...
   ... 227 URLs únicas coletadas ...
   ... 239 URLs únicas coletadas ...
   ... 251 URLs únicas coletadas ...
   ... 263 URLs únicas coletadas ...
   ... 275 URLs únicas coletadas ...
   ... 287 URLs únicas coletadas ...
   ... 299 URLs únicas coletadas ...
   ... 311 URLs únicas coletadas ...
   ... 323 URLs únicas coleta

In [7]:
# Célula 6: Fase 2 - Visita cada URL e coleta os dados detalhados (VERSÃO CORRIGIDA)

lista_de_dados = [] # Lista para armazenar um dicionário por imóvel
total_urls = len(urls_para_visitar)

print(f"Iniciando Fase 2: Visitando {total_urls} páginas...")

for i, url_imovel in enumerate(urls_para_visitar):
    
    print(f"Processando {i+1}/{total_urls}: {url_imovel}")
    
    try:
        driver.get(url_imovel)
        time.sleep(7) # Espera a página carregar
        
        soup_imovel = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Dicionário para este imóvel
        imovel_data = {'URL': url_imovel}
        
        # --- 1. Extração de Rua e Bairro (COM SEPARAÇÃO) ---
        imovel_data['Rua'] = np.nan
        imovel_data['Bairro'] = np.nan # Nova coluna
        imovel_data['Cidade'] = np.nan # Nova coluna
        
        try:
            # Encontra o contêiner do mapa
            map_wrapper = soup_imovel.find('div', {'data-testid': 'smallMapWrapper'})
            if map_wrapper:
                
                # Padrão Rua: <h4 class="... EqjlRj">Rua Newton Braga</h4>
                rua_tag = map_wrapper.find('h4', class_='EqjlRj')
                if rua_tag:
                    imovel_data['Rua'] = rua_tag.text.strip()
                
                # Padrão Bairro/Cidade: <small class="... pwAPLE">Vila Maria, São Paulo</small>
                bairro_tag = map_wrapper.find('small', class_='pwAPLE')
                if bairro_tag:
                    bairro_cidade_texto = bairro_tag.text.strip()
                    try:
                        # Tenta dividir a string "Bairro, Cidade"
                        partes = bairro_cidade_texto.split(',')
                        imovel_data['Bairro'] = partes[0].strip() # Ex: "Vila Maria"
                        imovel_data['Cidade'] = partes[1].strip() # Ex: "São Paulo"
                    except Exception:
                        # Se falhar (ex: não tiver vírgula), salva o texto todo em 'Bairro'
                        imovel_data['Bairro'] = bairro_cidade_texto
            
        except Exception as e:
            print(f"   - Erro ao processar bloco de endereço: {e}")
            
        
        # --- 2. Extração de Custos (Sem alteração) ---
        imovel_data['Aluguel'] = np.nan
        imovel_data['Condominio'] = np.nan
        imovel_data['IPTU'] = np.nan
        imovel_data['Total'] = np.nan
        
        try:
            price_table = soup_imovel.find('ul', {'data-testid': 'listing-price-table'})
            if price_table:
                list_items = price_table.find_all('li')
                
                for item in list_items:
                    text_parts = [t.text.strip() for t in item.find_all(['p', 'h4'])]
                    
                    if len(text_parts) >= 2:
                        label = text_parts[0]
                        value = text_parts[-1] 
                        
                        if 'Aluguel' in label:
                            imovel_data['Aluguel'] = value
                        elif 'Condomínio' in label:
                            imovel_data['Condominio'] = value
                        elif 'IPTU' in label:
                            imovel_data['IPTU'] = value
                        elif 'Total' in label:
                            imovel_data['Total'] = value
        except Exception as e:
            print(f"   - Erro ao processar tabela de preços: {e}")

            
        # --- 3. Extração de Especificações (MÉTODO INTELIGENTE - SEM ERROS DE ORDEM) ---
        imovel_data['Metragem'] = np.nan
        imovel_data['Quartos'] = np.nan
        imovel_data['Banheiros'] = np.nan
        imovel_data['Vagas'] = np.nan
        imovel_data['Andar'] = np.nan
        imovel_data['Pet'] = np.nan
        imovel_data['Mobilia'] = np.nan

        try:
            main_info = soup_imovel.find('div', {'data-testid': 'house-main-info'})
            if main_info:
                spec_wrappers = main_info.find_all('div', class_='MainInfo_iconDescriptionWrapper__St8RA')
                
                # Loop por CADA item, lendo o texto para identificar
                for spec in spec_wrappers:
                    texto_spec = spec.find('p').text.strip()
                    
                    if 'm²' in texto_spec:
                        imovel_data['Metragem'] = texto_spec
                    elif 'quarto' in texto_spec:
                        imovel_data['Quartos'] = texto_spec
                    elif 'banheiro' in texto_spec:
                        imovel_data['Banheiros'] = texto_spec
                    elif 'vaga' in texto_spec:
                        imovel_data['Vagas'] = texto_spec
                    elif 'andar' in texto_spec:
                        imovel_data['Andar'] = texto_spec
                    elif 'pet' in texto_spec:
                        imovel_data['Pet'] = texto_spec
                    elif 'mobília' in texto_spec:
                        imovel_data['Mobilia'] = texto_spec

        except Exception as e:
            print(f"   - Erro ao processar especificações: {e}")

            
        # --- 4. Extração de Comodidades (Sem alteração) ---
        try:
            amenities_div = soup_imovel.find('div', {'data-testid': 'amenities'})
            tags = amenities_div.find_all(['p', 'span'])
            comodidades = [tag.text.strip() for tag in tags if tag.text.strip()]
            imovel_data['Comodidades'] = ", ".join(comodidades)
        except:
            imovel_data['Comodidades'] = np.nan
        
        # --- Fim da Coleta ---
        
        lista_de_dados.append(imovel_data)
        
        # Backup parcial a cada 50 imóveis
        if (i+1) % 50 == 0:
            print(f"--- Salvando backup parcial com {i+1} imóveis ---")
            pd.DataFrame(lista_de_dados).to_csv('backup_imoveis_quintoandar.csv', index=False, encoding='utf-8-sig')

    except Exception as e:
        print(f"   !!! ERRO GERAL ao processar a URL {url_imovel}: {e} !!!")
        continue

print("\nFase 2 Concluída. Todos os imóveis foram visitados.")

Iniciando Fase 2: Visitando 909 páginas...
Processando 1/909: https://www.quintoandar.com.br/imovel/894705605/alugar/apartamento-3-quartos-vila-regente-feijo-sao-paulo?from_route=%22search_results%22&house_tags=exclusivity&house_tags=rentPriceDecreased&search_id=%22b6db7d47-f35f-44a1-912c-cf17bf5ec3d9%22&search_rank=%7B%22sortMode%22%3A%22relevance%22%2C%22searchMode%22%3A%22list%22%2C%22resultsOrigin%22%3A%22search%22%2C%22rank%22%3A872%2C%22personalization%22%3Atrue%7D
Processando 2/909: https://www.quintoandar.com.br/imovel/895278176/alugar/apartamento-2-quartos-lapa-sao-paulo?from_route=%22search_results%22&house_tags=newAd&search_id=%22b6db7d47-f35f-44a1-912c-cf17bf5ec3d9%22&search_rank=%7B%22sortMode%22%3A%22relevance%22%2C%22searchMode%22%3A%22list%22%2C%22resultsOrigin%22%3A%22search%22%2C%22rank%22%3A6%2C%22personalization%22%3Atrue%7D
Processando 3/909: https://www.quintoandar.com.br/imovel/893060689/alugar/apartamento-1-quarto-mirandopolis-sao-paulo?from_route=%22search_resu

In [ ]:
# Célula 7: Criar o DataFrame final e salvar em CSV

print("Criando DataFrame final...")
df_final = pd.DataFrame(lista_de_dados)

# Limpeza opcional: remover linhas onde o Aluguel não foi encontrado
df_final = df_final.dropna(subset=['Aluguel'])

print(f"Total de {len(df_final)} imóveis coletados com sucesso.")
print("Amostra dos dados:")
print(df_final.head())

# Salvar o arquivo CSV final
nome_arquivo = 'imoveis_quintoandar_NIVEL_2.csv'
df_final.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')

print(f"\nDataset completo salvo com sucesso em: {nome_arquivo}")

Criando DataFrame final...
Total de 907 imóveis coletados com sucesso.
Amostra dos dados:
                                                 URL                     Rua  \
0  https://www.quintoandar.com.br/imovel/89470560...   Rua Rodrigues Barbosa   
1  https://www.quintoandar.com.br/imovel/89527817...  Rua Coelho de Carvalho   
2  https://www.quintoandar.com.br/imovel/89306068...       Avenida Jabaquara   
3  https://www.quintoandar.com.br/imovel/89305391...    Rua Francisco Leitão   
4  https://www.quintoandar.com.br/imovel/89463101...             Rua Antenas   

               Bairro     Cidade   Aluguel Condominio    IPTU      Total  \
0  Vila Regente Feijó  São Paulo  R$ 4.000   R$ 1.200  R$ 250   R$ 5.604   
1                Lapa  São Paulo  R$ 8.500   R$ 1.995   R$ 50  R$ 10.873   
2        Mirandópolis  São Paulo  R$ 2.800     R$ 550   R$ 25   R$ 3.483   
3           Pinheiros  São Paulo  R$ 4.800     R$ 745    R$ 0   R$ 5.730   
4     Vila California  São Paulo  R$ 1.850     R$

: 